# Example 03 — Two-DOF Chain of Oscillators with Unilateral Spring

**Reference**: `matlab/NLvib/EXAMPLES/03_twoDOFoscillator_unilateralSpring/`

**Metric**: `a_rms = sqrt(sum(Q(1:2:end,:).^2)) / sqrt(2)` — RMS of DOF 0 across all harmonics.

**System**: 2-DOF chain, `mi=[1,1]`, `ki=[0,1,1]`, `di=0.03*ki`, unilateral spring k=100 gap=1 at DOF 0, force F=0.1 at DOF 1, H=21.

In [ ]:
from __future__ import annotations

import subprocess
import shutil
import sys
from pathlib import Path
from IPython.display import Image, display

import numpy as np
import scipy.io
import matplotlib.pyplot as plt

repo_root  = Path('/Users/julianjurai/Desktop/CustomApps/nonlinear_vibration_analysis_toolbox')
src_path   = str(repo_root / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

script_dir = repo_root / 'matlab/NLvib/EXAMPLES/03_twoDOFoscillator_unilateralSpring'
print('Repo root:', repo_root)
print('Script dir:', script_dir)

## MATLAB / Octave

In [ ]:
octave_bin = shutil.which('octave')
if not octave_bin:
    raise RuntimeError(
        "Octave not found on PATH. Install Octave and ensure it is on your PATH. "
        "See https://octave.org/download for installation instructions."
    )
print(f'Using Octave: {octave_bin}')

_hb_mat = script_dir / 'hb_data.mat'
_frf_png = script_dir / 'matlab_frf.png'
if _hb_mat.exists() and _frf_png.exists():
    print('Octave outputs already exist — skipping re-run.')
else:
    result = subprocess.run(
        [octave_bin, '--no-gui', 'save_data.m'],
        cwd=str(script_dir), check=True, capture_output=True, text=True, timeout=600
    )
    print('Octave stdout (last 500 chars):', result.stdout[-500:])
    if result.stderr:
        print('STDERR (last 200 chars):', result.stderr[-200:])
display(Image(filename=str(_frf_png)))

## Python

In [ ]:
from nlvib.systems.oscillators import ChainOfOscillators
from nlvib.nonlinearities.elements import unilateral_spring
from nlvib.solvers.harmonic_balance import hb_residual
from nlvib.continuation.solver import ContinuationSolver, ContinuationOptions
import scipy.linalg as _la_hb

MASSES             = [1.0, 1.0]
STIFFNESSES        = [0.0, 1.0, 1.0]
DAMPINGS           = [0.0, 0.03, 0.03]
CONTACT_STIFFNESS  = 100.0
CONTACT_GAP        = 1.0
CONTACT_DOF        = 0
EXCITATION_DOF     = 1
EXCITATION_AMP     = 0.1
H                  = 21
OMEGA_START        = 0.5
OMEGA_END          = 0.8

system = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
nl_element = unilateral_spring(k=CONTACT_STIFFNESS, gap=CONTACT_GAP, dof_index=CONTACT_DOF)
system.add_nonlinear_element(nl_element)

n_dof      = system.n_dof
n_total    = n_dof * (2 * H + 1)
excitation = {'dof': EXCITATION_DOF, 'amplitude': EXCITATION_AMP}

# Start at OMEGA_END=0.8 with linear initial guess (matches MATLAB Om_s=0.8 -> Om_e=0.5)
M_d = system.M.toarray(); D_d = system.D.toarray(); K_d = system.K.toarray()
Fex_c = np.zeros(n_dof, dtype=complex)
Fex_c[EXCITATION_DOF] = EXCITATION_AMP
Om_start_hb = OMEGA_END  # 0.8
Q1_lin = _la_hb.solve(-Om_start_hb**2 * M_d + 1j * Om_start_hb * D_d + K_d, Fex_c)
Q = np.zeros(n_total)
Q[(2*1-1)*n_dof : (2*1-1)*n_dof + n_dof] = Q1_lin.real
Q[2*1*n_dof     : 2*1*n_dof     + n_dof] = -Q1_lin.imag

R, J = hb_residual(Q, Om_start_hb, system, H, excitation)
print(f'n_dof={n_dof}, H={H}, n_total={n_total}')
print(f'Initial residual at omega={Om_start_hb:.3f}: {np.linalg.norm(R):.3e}')

# Arc-length continuation with negated lambda to trace from OMEGA_END down to OMEGA_START
# ds_min=0.01 avoids infinite step-collapse loops at folds (ds_min 1e-6 caused 900s+ hangs)
def neg_hb_residual(q, neg_om):
    return hb_residual(q, -neg_om, system, H, excitation)

opts = ContinuationOptions(
    verbose=False,
    ds_initial=0.02,
    ds_min=0.01,
    ds_max=0.06,
    max_steps=150,
    max_newton_iter=10,
    newton_tol=1e-8,
    adapt_step=True,
    lambda_min=-(OMEGA_END + 0.05),
    lambda_max=-(OMEGA_START - 0.05),
)

result = ContinuationSolver().run(neg_hb_residual, Q, -Om_start_hb, opts)
print(f'Continuation: {result.message}')
print(f'Steps accepted: {result.n_steps}')

# Negate lambda back to recover omega
solutions  = result.solutions
omega_py   = -solutions[:, -1]
Q_all_py   = solutions[:, :-1]
Q_dof0_py  = Q_all_py.reshape(Q_all_py.shape[0], 2*H+1, n_dof)[:, :, 0]
a_rms_py   = np.sqrt(np.sum(Q_dof0_py**2, axis=1)) / np.sqrt(2)
mask       = (omega_py >= OMEGA_START) & (omega_py <= OMEGA_END)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omega_py[mask], a_rms_py[mask], 'b-', linewidth=2)
ax.set_xlabel('Excitation frequency (rad/s)')
ax.set_ylabel('a_rms (DOF 0, all harmonics)')
ax.set_title('Example 03 -- Two-DOF Unilateral Spring: FRF (Python HB)')
ax.set_xlim(OMEGA_START, OMEGA_END)
ax.grid(True, linestyle='--', alpha=0.5)
fig.tight_layout()

## Results — Overlay: MATLAB HB + Python HB + Python Shooting (Floquet stability)

In [ ]:
# ── Cell 3: Shooting continuation + Floquet stability ──────────────────────────
from nlvib.solvers.shooting import shooting_residual, newmark_step
import scipy.linalg as _la

# ── System matrices (dense, for Newmark) ──
M_d = system.M.toarray()
D_d = system.D.toarray()
K_d = system.K.toarray()

# ── Excitation vector ──
Fex_vec = np.zeros(n_dof); Fex_vec[EXCITATION_DOF] = EXCITATION_AMP

def f_ext(t, om):
    """Harmonic excitation F·cos(Ω·t)."""
    return Fex_vec * np.cos(om * t)

# ── Shooting resolution ──
N_SHOOT = 32   # steps per period

# ── Pre-converge shooting solution at OMEGA_END=0.8 ──
Om_s = OMEGA_END   # 0.8
Fex_c = np.zeros(n_dof, dtype=complex); Fex_c[EXCITATION_DOF] = EXCITATION_AMP
Q1_lin = _la.solve(-Om_s**2 * M_d + 1j*Om_s*D_d + K_d, Fex_c)
ys0 = np.concatenate([Q1_lin.real, -Om_s * Q1_lin.imag])

y_cur = ys0.copy()
for it in range(40):
    R_s, J_s = shooting_residual(
        y_cur, Om_s, system, n_periods=1, n_steps=N_SHOOT,
        f_ext_fn=lambda t: f_ext(t, Om_s)
    )
    nrm = np.linalg.norm(R_s)
    if nrm < 1e-8:
        break
    try:
        dy = np.linalg.solve(J_s, -R_s)
    except np.linalg.LinAlgError:
        dy = np.linalg.lstsq(J_s, -R_s, rcond=None)[0]
    y_cur = y_cur + dy
print(f'Shooting pre-converged at Ω={Om_s}: |R|={np.linalg.norm(R_s):.3e}')

# ── Negated-lambda residual: solver sees λ = -Ω so it traces decreasing Ω ──
def neg_shoot_residual(y, neg_om):
    om = -neg_om
    return shooting_residual(
        y, om, system, n_periods=1, n_steps=N_SHOOT,
        f_ext_fn=lambda t: f_ext(t, om)
    )

shoot_opts = ContinuationOptions(
    verbose=False,
    ds_initial=0.025,
    ds_min=1e-4,
    ds_max=0.02,
    max_steps=60,
    max_newton_iter=20,
    newton_tol=1e-7,
    adapt_step=True,
    lambda_min=-(OMEGA_END + 0.05),
    lambda_max=-(OMEGA_START - 0.05),
)

shoot_result = ContinuationSolver().run(
    neg_shoot_residual, y_cur, -Om_s, shoot_opts
)
print(f'Shooting continuation: {shoot_result.message}')
print(f'Steps accepted: {shoot_result.n_steps}')

shoot_solutions = shoot_result.solutions     # shape (n_steps, 2*n_dof + 1)
omega_shoot = -shoot_solutions[:, -1]        # negate back
Ys_all = shoot_solutions[:, :-1]            # shape (n_steps, 2*n_dof)

# ── Floquet stability: evaluate monodromy matrix at each shooting solution ──
print('Running Floquet stability analysis...')
n_state = 2 * n_dof
a_shoot = np.zeros(len(omega_shoot))
stable_shoot = np.zeros(len(omega_shoot), dtype=bool)
mucrit_shoot = np.zeros(len(omega_shoot), dtype=complex)

def fnl_wrap(q, dq):
    """Wrap system nonlinear forces for Newmark step."""
    return system.eval_nonlinear_forces(q, dq)[0]

for i, (om, ys) in enumerate(zip(omega_shoot, Ys_all)):
    R_i, J_i = shooting_residual(
        ys, om, system, n_periods=1, n_steps=N_SHOOT,
        f_ext_fn=lambda t, _om=om: f_ext(t, _om)
    )
    monodromy = J_i + np.eye(n_state)

    # Reconstruct DOF-1 time series via Newmark for FFT amplitude
    T_i = 2.0 * np.pi / om
    dt_i = T_i / N_SHOOT
    q_traj = np.zeros(N_SHOOT)
    y_ts = ys.copy()
    ddq_ts = None
    for k in range(N_SHOOT):
        t_k1 = (k + 1) * dt_i
        f_k1 = f_ext(t_k1, om)
        y_ts, ddq_ts = newmark_step(
            y_ts, f_k1, M_d, D_d, K_d, fnl_wrap, dt_i, ddq_n=ddq_ts
        )
        q_traj[k] = y_ts[n_dof - 1]   # DOF 1

    Qc_fft = np.fft.fft(q_traj) / N_SHOOT
    a_shoot[i] = 2.0 * abs(Qc_fft[1])

    evals = np.linalg.eigvals(monodromy)
    mu_lead = evals[np.argmax(np.abs(evals))]
    mucrit_shoot[i] = mu_lead
    stable_shoot[i] = abs(mu_lead) <= 1.0 + 1e-6

n_stable = int(stable_shoot.sum())
n_unstable = int((~stable_shoot).sum())
print(f'Stable solutions: {n_stable}/{len(stable_shoot)}')
print(f'Unstable solutions: {n_unstable}/{len(stable_shoot)}')

In [ ]:
# ── Cell 4a: Display MATLAB / Octave figure ─────────────────────────────────────
from IPython.display import Image, display
display(Image(filename=str(script_dir / 'matlab_frf.png')))

In [ ]:
# ── Cell 4b: Python HB + Shooting + Floquet stability plot ──────────────────────
import scipy.io as _sio

mat_data        = _sio.loadmat(script_dir / 'hb_data.mat')
Om_HB_matlab    = mat_data['Om_HB'].ravel()
a_rms_HB_matlab = mat_data['a_rms_HB'].ravel()

# Python HB fundamental harmonic amplitude of DOF 1
# Layout (blocks of n_dof): cosine h=1 → (2*1-1)*n_dof + 1, sine h=1 → 2*1*n_dof + 1
c_idx = (2*1 - 1) * n_dof + 1   # = 3
s_idx =  2*1      * n_dof + 1   # = 5
a_HB_py = np.sqrt(Q_all_py[:, c_idx]**2 + Q_all_py[:, s_idx]**2)

# Filter to [OMEGA_START, OMEGA_END]
mask_m  = (Om_HB_matlab >= OMEGA_START) & (Om_HB_matlab <= OMEGA_END)
mask_py = (omega_py    >= OMEGA_START) & (omega_py    <= OMEGA_END)
mask_sh = (omega_shoot >= OMEGA_START) & (omega_shoot <= OMEGA_END)

# Detect bifurcation points from Floquet multiplier angle changes
n_sh = len(omega_shoot)
bp_oms, bp_as, bp_types = [], [], []
for i in range(1, n_sh):
    if stable_shoot[i] != stable_shoot[i - 1]:
        bp_om = (omega_shoot[i] + omega_shoot[i - 1]) / 2.0
        bp_a  = (a_shoot[i]    + a_shoot[i - 1])    / 2.0
        mu_i  = mucrit_shoot[i]
        ang   = abs(np.angle(mu_i))
        if ang < 1e-2:
            btype = 'TP'
        elif abs(ang - np.pi) < 1e-2:
            btype = 'PD'
        else:
            btype = 'NS'
        bp_oms.append(bp_om); bp_as.append(bp_a); bp_types.append(btype)

# ── Python-only plot ──
fig, ax = plt.subplots(figsize=(9, 5))

# Python HB fundamental harmonic of DOF 1
ax.plot(omega_py[mask_py], a_HB_py[mask_py],
        'b-', linewidth=2.0, label='Python HB (fund. harm. DOF 1)')

# Python Shooting stable / unstable
stab_mask   = mask_sh & stable_shoot
unstab_mask = mask_sh & ~stable_shoot
if stab_mask.any():
    ax.plot(omega_shoot[stab_mask], a_shoot[stab_mask],
            'k.', markersize=5, label='Python Shooting (stable)')
if unstab_mask.any():
    ax.plot(omega_shoot[unstab_mask], a_shoot[unstab_mask],
            'rx', markersize=7, markeredgewidth=1.5, label='Python Shooting (unstable)')

# Bifurcation points
bp_style = {'TP': ('ro', 'r', 'TP (fold)'),
             'PD': ('ms', 'm', 'PD (period-doubling)'),
             'NS': ('c^', 'c', 'NS (Neimark-Sacker)')}
plotted_types = set()
for om_bp, a_bp, btype in zip(bp_oms, bp_as, bp_types):
    marker, color, label = bp_style.get(btype, ('ko', 'k', btype))
    lbl = label if btype not in plotted_types else None
    ax.plot(om_bp, a_bp, marker, markersize=10, color=color,
            markerfacecolor=color, label=lbl)
    plotted_types.add(btype)

ax.set_xlabel('excitation frequency')
ax.set_ylabel('response amplitude')
ax.set_title('Example 03 — Two-DOF Unilateral Spring\n'
             'Python HB · Python Shooting + Floquet stability',
             fontsize=11)
ax.set_xlim(0.5, 0.8)  # matches MATLAB xlim(sort([Om_s Om_e]))
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
fig.tight_layout()
plt.show()
print(f'Bifurcation points detected: {len(bp_oms)}')
for om_bp, a_bp, btype in zip(bp_oms, bp_as, bp_types):
    print(f'  {btype}: Ω = {om_bp:.4f}, a = {a_bp:.6f}')

In [ ]:
# ── Cell 5: Assertion — peak amplitude error < 5% ──────────────────────────────
# Compare Python HB fundamental harmonic (DOF 1) against MATLAB HB a_rms (DOF 0).
# Both are computed on the same frequency axis and both should agree within 5%.
#
# MATLAB's exported metric is a_rms (DOF 0); Python HB uses a_HB (DOF 1 fund.).
# We compare a_rms_py (DOF 0, same metric as MATLAB) for apples-to-apples.
# The a_rms_py was already computed in Cell 2.

omega_m_f  = Om_HB_matlab[mask_m]
a_rms_m_f  = a_rms_HB_matlab[mask_m]
omega_py_f = omega_py[mask_py]
a_rms_py_f = a_rms_py[mask_py]

peak_matlab       = float(a_rms_m_f.max())
peak_py           = float(a_rms_py_f.max())
peak_matlab_omega = float(omega_m_f[a_rms_m_f.argmax()])
peak_py_omega     = float(omega_py_f[a_rms_py_f.argmax()])
rel_err = abs(peak_py - peak_matlab) / peak_matlab

# Shooting peak (DOF 1, fundamental harmonic)
if mask_sh.any():
    a_shoot_f = a_shoot[mask_sh]
    om_shoot_f = omega_shoot[mask_sh]
    peak_shoot = float(a_shoot_f.max())
    peak_shoot_omega = float(om_shoot_f[a_shoot_f.argmax()])
else:
    peak_shoot = float('nan')
    peak_shoot_omega = float('nan')

print('=' * 62)
print('  Peak Amplitude Comparison — Example 03')
print('=' * 62)
print(f'  {"":22} {"MATLAB/Octave":>14} {"Python HB":>10} {"Shooting":>10}')
print(f'  {"-"*56}')
print(f'  {"Peak a_rms (DOF 0)":<22} {peak_matlab:>14.6f} {peak_py:>10.6f} {"—":>10}')
print(f'  {"Peak omega (rad/s)":<22} {peak_matlab_omega:>14.4f} {peak_py_omega:>10.4f} {peak_shoot_omega:>10.4f}')
print(f'  {"HB rel. error":<22} {"—":>14} {rel_err:>10.4f} {"—":>10}')
print('=' * 62)

assert rel_err < 0.05, (
    f'Peak amplitude relative error {rel_err:.4f} exceeds 5% threshold. '
    f'Python HB peak={peak_py:.6f}, MATLAB peak={peak_matlab:.6f}'
)
print(f'PASS: HB peak error = {rel_err*100:.2f}% < 5%')
print(f'Shooting peak (fund. harm. DOF 1) = {peak_shoot:.6f} at Ω = {peak_shoot_omega:.4f}')

## MATLAB vs Python

In [ ]:
# ── Cell MVS-1: Side-by-side figure — MATLAB (image) vs Python HB + Floquet ──
import matplotlib.patches as mpatches

matlab_img_mvs = plt.imread(str(script_dir / 'matlab_frf.png'))

fig_mvs, axes_mvs = plt.subplots(1, 2, figsize=(15, 5))

# ── Left panel: MATLAB / Octave ──
axes_mvs[0].imshow(matlab_img_mvs)
axes_mvs[0].axis('off')
axes_mvs[0].set_title('MATLAB / Octave (reference)', fontsize=12, fontweight='bold')

# ── Right panel: Python HB with Floquet colouring ──
ax_py_mvs = axes_mvs[1]

# Compute a_rms equivalent for HB so both panels show the same metric (DOF 0, all harmonics)
mask_py_mvs = (omega_py >= OMEGA_START) & (omega_py <= OMEGA_END)
mask_sh_mvs = (omega_shoot >= OMEGA_START) & (omega_shoot <= OMEGA_END)

# Python HB a_rms (DOF 0) — already computed as a_rms_py
ax_py_mvs.plot(omega_py[mask_py_mvs], a_rms_py[mask_py_mvs],
               color='steelblue', linewidth=2.0, label='Python HB (a_rms DOF 0)', zorder=3)

# Shooting: stable = solid line, unstable = dashed line
if mask_sh_mvs.any():
    om_sh_f  = omega_shoot[mask_sh_mvs]
    a_sh_f   = a_shoot[mask_sh_mvs]
    stab_f   = stable_shoot[mask_sh_mvs]

    # Plot stable segments
    # Build run-length encoded segments
    def _plot_segments(ax, x, y, flag, color, style, label, zorder=2):
        lbl_done = False
        i = 0
        while i < len(x):
            j = i
            while j < len(x) and flag[j] == flag[i]:
                j += 1
            if flag[i]:
                lbl = label if not lbl_done else None
                ax.plot(x[i:j], y[i:j], linestyle=style, color=color,
                        linewidth=1.5, label=lbl, zorder=zorder)
                lbl_done = True
            i = j

    _plot_segments(ax_py_mvs, om_sh_f, a_sh_f, stab_f,
                   color='darkorange', style='-', label='Shooting (stable)')
    _plot_segments(ax_py_mvs, om_sh_f, a_sh_f, ~stab_f,
                   color='darkorange', style='--', label='Shooting (unstable)')

# Bifurcation points
bp_colors = {'TP': 'red', 'PD': 'magenta', 'NS': 'cyan'}
bp_markers = {'TP': 'o', 'PD': 's', 'NS': '^'}
plotted_bp_mvs = set()
for om_bp, a_bp, btype in zip(bp_oms, bp_as, bp_types):
    if OMEGA_START <= om_bp <= OMEGA_END:
        lbl = f'{btype} (bifurcation)' if btype not in plotted_bp_mvs else None
        ax_py_mvs.plot(om_bp, a_bp,
                       marker=bp_markers.get(btype, 'D'),
                       color=bp_colors.get(btype, 'black'),
                       markersize=10, linewidth=0, label=lbl, zorder=5)
        plotted_bp_mvs.add(btype)

ax_py_mvs.set_xlabel('Excitation frequency (rad/s)', fontsize=10)
ax_py_mvs.set_ylabel('a_rms (DOF 0, all harmonics)', fontsize=10)
ax_py_mvs.set_title('Python HB + Shooting (stable=solid, unstable=dashed)', fontsize=12, fontweight='bold')

# Match x/y limits — use linear scale (no log)
xlim_mvs = (OMEGA_START, OMEGA_END)
ylim_mvs = (0.0, max(float(a_rms_py[mask_py_mvs].max()),
                     float(a_shoot[mask_sh_mvs].max()) if mask_sh_mvs.any() else 0) * 1.15)
ax_py_mvs.set_xlim(*xlim_mvs)
ax_py_mvs.set_ylim(*ylim_mvs)
ax_py_mvs.legend(fontsize=8, loc='upper left')
ax_py_mvs.grid(True, linestyle='--', alpha=0.4)

fig_mvs.suptitle('Example 03 — Two-DOF Unilateral Spring: MATLAB vs Python', fontsize=13)
fig_mvs.tight_layout()
plt.show()

if bp_oms:
    print(f'Bifurcation points detected: {len(bp_oms)}')
    for om_bp, a_bp, btype in zip(bp_oms, bp_as, bp_types):
        print(f'  {btype}: Omega={om_bp:.4f} rad/s, a={a_bp:.6f}')
else:
    print('No bifurcation points detected in the shooting frequency range.')

In [ ]:
# ── Cell MVS-2: Comparison metrics table ──────────────────────────────────────
import math

# Re-use filtered arrays from prior cells
omega_m_f_mvs  = Om_HB_matlab[mask_m]
a_rms_m_f_mvs  = a_rms_HB_matlab[mask_m]
omega_py_f_mvs = omega_py[mask_py]
a_rms_py_f_mvs = a_rms_py[mask_py]

# ── Peak amplitude (a_rms DOF 0) ──
peak_amp_m   = float(a_rms_m_f_mvs.max())
peak_amp_py  = float(a_rms_py_f_mvs.max())
diff_amp     = abs(peak_amp_py - peak_amp_m)
pct_amp      = 100.0 * diff_amp / peak_amp_m

# ── Peak frequency ──
peak_om_m    = float(omega_m_f_mvs[a_rms_m_f_mvs.argmax()])
peak_om_py   = float(omega_py_f_mvs[a_rms_py_f_mvs.argmax()])
diff_om      = abs(peak_om_py - peak_om_m)
pct_om       = 100.0 * diff_om / peak_om_m

# ── Number of continuation steps ──
steps_m      = len(omega_m_f_mvs)
steps_py     = result.n_steps   # Python HB continuation object
diff_steps   = abs(steps_py - steps_m)
pct_steps    = 100.0 * diff_steps / max(steps_m, 1)

# ── Frequency range ──
frange_m_lo  = float(omega_m_f_mvs.min())
frange_m_hi  = float(omega_m_f_mvs.max())
frange_py_lo = float(omega_py_f_mvs.min())
frange_py_hi = float(omega_py_f_mvs.max())
diff_lo      = abs(frange_py_lo - frange_m_lo)
diff_hi      = abs(frange_py_hi - frange_m_hi)
pct_lo       = 100.0 * diff_lo / max(abs(frange_m_lo), 1e-12)
pct_hi       = 100.0 * diff_hi / max(abs(frange_m_hi), 1e-12)

hdr = f"{'Metric':<30} {'MATLAB':>12} {'Python':>12} {'|Diff|':>12} {'Rel.err %':>10}"
sep = '-' * len(hdr)
print(hdr)
print(sep)
print(f"{'Peak amplitude (a_rms)':<30} {peak_amp_m:>12.6f} {peak_amp_py:>12.6f} {diff_amp:>12.6f} {pct_amp:>9.3f}%")
print(f"{'Peak frequency (rad/s)':<30} {peak_om_m:>12.4f} {peak_om_py:>12.4f} {diff_om:>12.4f} {pct_om:>9.3f}%")
print(f"{'N continuation steps':<30} {steps_m:>12d} {steps_py:>12d} {diff_steps:>12d} {pct_steps:>9.2f}%")
print(f"{'Freq range — low (rad/s)':<30} {frange_m_lo:>12.4f} {frange_py_lo:>12.4f} {diff_lo:>12.4f} {pct_lo:>9.3f}%")
print(f"{'Freq range — high (rad/s)':<30} {frange_m_hi:>12.4f} {frange_py_hi:>12.4f} {diff_hi:>12.4f} {pct_hi:>9.3f}%")

In [ ]:
# ── Cell MVS-3: Runtime comparison — Python (timed inline) vs Octave (stdout) ──
import time
import re

# ── Python HB runtime (fresh run, same parameters) ──
import scipy.linalg as _la_rt

_t0_py = time.perf_counter()

system_rt = ChainOfOscillators(masses=MASSES, stiffnesses=STIFFNESSES, dampings=DAMPINGS)
nl_rt     = unilateral_spring(k=CONTACT_STIFFNESS, gap=CONTACT_GAP, dof_index=CONTACT_DOF)
system_rt.add_nonlinear_element(nl_rt)

n_total_rt = n_dof * (2 * H + 1)
excitation_rt = {'dof': EXCITATION_DOF, 'amplitude': EXCITATION_AMP}

M_rt = system_rt.M.toarray(); D_rt = system_rt.D.toarray(); K_rt = system_rt.K.toarray()
Fex_c_rt = np.zeros(n_dof, dtype=complex)
Fex_c_rt[EXCITATION_DOF] = EXCITATION_AMP
Q_rt = np.zeros(n_total_rt)
Q1_rt = _la_rt.solve(-OMEGA_END**2 * M_rt + 1j * OMEGA_END * D_rt + K_rt, Fex_c_rt)
Q_rt[(2*1-1)*n_dof : (2*1-1)*n_dof + n_dof] = Q1_rt.real
Q_rt[2*1*n_dof     : 2*1*n_dof     + n_dof] = -Q1_rt.imag

def _neg_hb_rt(q, neg_om):
    return hb_residual(q, -neg_om, system_rt, H, excitation_rt)

opts_rt = ContinuationOptions(
    verbose=False, ds_initial=0.02, ds_min=0.01, ds_max=0.06,
    max_steps=150, max_newton_iter=10, newton_tol=1e-8,
    adapt_step=True,
    lambda_min=-(OMEGA_END + 0.05), lambda_max=-(OMEGA_START - 0.05),
)
_result_rt = ContinuationSolver().run(_neg_hb_rt, Q_rt, -OMEGA_END, opts_rt)

_t1_py = time.perf_counter()
t_python_s = _t1_py - _t0_py

# ── Octave runtime — re-run with timing ──
_t0_oct = time.perf_counter()
_oct_result = subprocess.run(
    [octave_bin, '--no-gui', 'save_data.m'],
    cwd=str(script_dir), capture_output=True, text=True, timeout=600
)
_t1_oct = time.perf_counter()
t_octave_s = _t1_oct - _t0_oct  # wall-clock (includes interpreter startup)

# Try to extract Octave's self-reported elapsed time from stdout
_oct_stdout = _oct_result.stdout
_m = re.search(r'[Ee]lapsed[^:\d]*[:=\s]+([0-9]+\.?[0-9]*)', _oct_stdout)
t_octave_self = float(_m.group(1)) if _m else None

speedup = t_octave_s / t_python_s if t_python_s > 0 else float('nan')

print(f'Python HB wall time   : {t_python_s:.2f} s  ({_result_rt.n_steps} steps)')
print(f'Octave wall time      : {t_octave_s:.2f} s  (includes interpreter startup)')
if t_octave_self is not None:
    print(f'Octave self-reported  : {t_octave_self:.2f} s')
print(f'Speedup (wall-clock)  : {speedup:.1f}x  (Python {"faster" if speedup > 1 else "slower"})')
if t_octave_self is not None:
    speedup_self = t_octave_self / t_python_s
    print(f'Speedup (Octave self) : {speedup_self:.1f}x')

In [ ]:
# ── Cell MVS-4: Harmonic content bar chart — Q1, Q3, Q5 at peak ──────────────
# Extract harmonic amplitudes at the peak-response index for MATLAB and Python.
#
# MATLAB Q_HB layout (from save_data.m): rows = harmonic coefficients (MATLAB ordering),
# columns = continuation steps. Each column is [DC; cos_h1; sin_h1; cos_h2; sin_h2; ...]
# for ALL DOFs interleaved: [DOF0_DC, DOF1_DC, DOF0_cos1, DOF1_cos1, DOF0_sin1, ...]
#
# Python Q_all_py layout: (n_steps, n_dof*(2H+1))
# Block layout: [DC_DOF0 DC_DOF1 | cos1_DOF0 cos1_DOF1 | sin1_DOF0 sin1_DOF1 | ...]
# => harmonic h, DOF d: cos index = (2h-1)*n_dof + d,  sin index = 2h*n_dof + d

Q_HB_matlab = mat_data['Q_HB']   # shape (n_coeff_matlab, n_steps_matlab)

# ── Find peak index in MATLAB ──
peak_idx_m  = int(a_rms_HB_matlab.argmax())
Q_peak_m    = Q_HB_matlab[:, peak_idx_m]   # shape (n_dof*(2H_matlab+1),)

# ── Find peak index in Python ──
peak_idx_py = int(a_rms_py.argmax())
Q_peak_py   = Q_all_py[peak_idx_py, :]   # shape (n_dof*(2H+1),)

H_mat_est   = (Q_HB_matlab.shape[0] // n_dof - 1) // 2   # infer H from shape
H_py        = H

def harmonic_amp_matlab(Q_col, h, dof, n_dof_=2):
    """Amplitude of harmonic h for DOF dof from MATLAB column (MATLAB interleaving)."""
    # MATLAB: [dc_row=0..n_dof-1, cos_h1=n_dof..2n_dof-1, sin_h1=2n_dof..3n_dof-1, ...]
    if h == 0:
        return abs(Q_col[dof])
    c_idx = (2*h - 1) * n_dof_ + dof
    s_idx =  2*h      * n_dof_ + dof
    return math.sqrt(Q_col[c_idx]**2 + Q_col[s_idx]**2)

def harmonic_amp_python(Q_row, h, dof, n_dof_=2):
    """Amplitude of harmonic h for DOF dof from Python row (Python interleaving)."""
    if h == 0:
        return abs(Q_row[dof])
    c_idx = (2*h - 1) * n_dof_ + dof
    s_idx =  2*h      * n_dof_ + dof
    return math.sqrt(Q_row[c_idx]**2 + Q_row[s_idx]**2)

harmonics = [1, 3, 5]
labels    = [f'Q{h}' for h in harmonics]
DOF_PLOT  = 0   # use DOF 0 (the one with unilateral contact — richest harmonic content)

amps_m  = [harmonic_amp_matlab(Q_peak_m,  h, DOF_PLOT, n_dof) for h in harmonics]
amps_py = [harmonic_amp_python(Q_peak_py, h, DOF_PLOT, n_dof) for h in harmonics]

x_pos   = np.arange(len(harmonics))
bar_w   = 0.35

fig_hc, ax_hc = plt.subplots(figsize=(7, 4))
bars_m  = ax_hc.bar(x_pos - bar_w/2, amps_m,  bar_w, label='MATLAB/Octave', color='forestgreen',  alpha=0.85)
bars_py = ax_hc.bar(x_pos + bar_w/2, amps_py, bar_w, label='Python HB',     color='royalblue', alpha=0.85)

ax_hc.set_xticks(x_pos)
ax_hc.set_xticklabels(labels, fontsize=11)
ax_hc.set_xlabel('Harmonic', fontsize=11)
ax_hc.set_ylabel('Amplitude (DOF 0)', fontsize=11)
ax_hc.set_title(f'Example 03 — Harmonic content at peak\n'
                f'(MATLAB Omega={Om_HB_matlab[peak_idx_m]:.4f}, '
                f'Python Omega={omega_py[peak_idx_py]:.4f} rad/s)', fontsize=10)
ax_hc.legend(fontsize=10)
ax_hc.grid(True, axis='y', linestyle='--', alpha=0.4)
fig_hc.tight_layout()
plt.show()

print(f'{"Harmonic":<12} {"MATLAB":>12} {"Python":>12} {"|Diff|":>12} {"Rel.err%":>10}')
print('-' * 58)
for h, am, ap in zip(harmonics, amps_m, amps_py):
    diff_h = abs(ap - am)
    pct_h  = 100.0 * diff_h / max(am, 1e-14)
    print(f'{"Q"+str(h):<12} {am:>12.6f} {ap:>12.6f} {diff_h:>12.6f} {pct_h:>9.3f}%')

In [ ]:
# ── Cell MVS-5: MOE / error assertions — all errors must be < 5% ──────────────
errors = {}

# 1. Peak amplitude (a_rms DOF 0)
errors['peak_amplitude'] = 100.0 * abs(peak_amp_py - peak_amp_m) / max(peak_amp_m, 1e-14)

# 2. Peak frequency
errors['peak_frequency'] = 100.0 * abs(peak_om_py - peak_om_m) / max(abs(peak_om_m), 1e-14)

# 3. Harmonic amplitudes Q1, Q3, Q5
for h, am, ap in zip(harmonics, amps_m, amps_py):
    key = f'harmonic_Q{h}'
    errors[key] = 100.0 * abs(ap - am) / max(am, 1e-14)

THRESHOLD_PCT = 5.0
print(f'Margin-of-error check (threshold = {THRESHOLD_PCT}%)')
print('=' * 55)
all_pass = True
for name, pct in errors.items():
    status = 'PASS' if pct < THRESHOLD_PCT else 'FAIL'
    if status == 'FAIL':
        all_pass = False
    print(f'  {name:<30} {pct:>8.3f}%  [{status}]')
print('=' * 55)

if all_pass:
    print(f'ALL {len(errors)} checks PASSED — Python HB matches MATLAB within {THRESHOLD_PCT}%.')
else:
    failing = [n for n, p in errors.items() if p >= THRESHOLD_PCT]
    raise AssertionError(
        f'{len(failing)} check(s) FAILED: {failing}. '
        f'Inspect cells above for discrepancies.'
    )